# Hillstrom Randomized Email Experiment — Reproducible Notebook

> **Reproducibility update:** This copy restores missing train/validation/test splits, explicitly trains the conversion T-Learner, fixes an ANCOVA contrast loop, and calculates Holm-adjusted A/B p-values. Outputs were cleared. Run top-to-bottom after placing the public CSV in `data/`. The original notebook and reported results are retained separately in the project report.

> The dataset was explored before an independent holdout was formalized. Treat subsequent test-set comparisons as exploratory rather than confirmatory.


Phase 1 — Understand the Business

We have a real life dataset Kevin Hillstorm's website, and all users have already been divided into three groups of segments for ab-test. The three segments are Womens E-mail, No-Email, and Mens E-Mail. Specifically, I am interested into Which campaign generates more incremental revenue? Which customers are most responsive to each campaign?

In [ ]:
import duckdb
import os
from pathlib import Path

# Place the public Hillstrom CSV in ./data or set HILLSTROM_CSV.
project_dir = Path.cwd()
manual_csv = os.environ.get("HILLSTROM_CSV")
if manual_csv:
    csv_path = Path(manual_csv).expanduser()
else:
    candidates = []
    for base in [project_dir / "data", project_dir.parent / "data", project_dir]:
        candidates.extend(sorted(base.glob("Kevin_Hillstrom*.csv")))
    if not candidates:
        raise FileNotFoundError(
            "Hillstrom CSV not found. Download it to ./data/ or set HILLSTROM_CSV."
        )
    csv_path = candidates[0]

con = duckdb.connect(database=":memory:")
con.execute("""
    CREATE OR REPLACE TABLE hillstrom_raw AS
    SELECT * FROM read_csv_auto(?)
""", [str(csv_path)])
print(f"Loaded: {csv_path}")


In [ ]:
con.sql("""
    SELECT *
    FROM hillstrom_raw
    LIMIT 10
""").df()



In [ ]:
con.sql("""
    SELECT COUNT(*) AS total_rows
    FROM hillstrom_raw
""").df()

In [ ]:
con.sql("""
    SELECT
        COUNT(*) AS Invalid_assignments
        FROM hillstrom_raw
        WHERE segment IS NULL
        OR segment NOT IN('Mens E-Mail',
                          'No E-Mail',
                          'Womens E-Mail')

""").df()

In [ ]:
con.sql("""
    SELECT
        COUNT(*) - COUNT(recency) AS missing_recency,
        COUNT(*) - COUNT(history_segment) AS missing_history_segment,
        COUNT(*) - COUNT(history) AS missing_history,
        COUNT(*) - COUNT(mens) AS missing_mens,
        COUNT(*) - COUNT(womens) AS missing_womens,
        COUNT(*) - COUNT(zip_code) AS missing_zip_code,
        COUNT(*) - COUNT(newbie) AS missing_newbie,
        COUNT(*) - COUNT(channel) AS missing_channel,
        COUNT(*) - COUNT(segment) AS missing_segment,
        COUNT(*) - COUNT(visit) AS missing_visit,
        COUNT(*) - COUNT(conversion) AS missing_conversion,
        COUNT(*) - COUNT(spend) AS missing_spend
    FROM hillstrom_raw
""").df()

In [ ]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE recency < 0)
            AS invalid_recency,

        COUNT(*) FILTER (WHERE history < 0)
            AS invalid_history,

        COUNT(*) FILTER (WHERE spend < 0)
            AS invalid_spend,

        COUNT(*) FILTER (WHERE mens NOT IN (0, 1))
            AS invalid_mens,

        COUNT(*) FILTER (WHERE womens NOT IN (0, 1))
            AS invalid_womens,

        COUNT(*) FILTER (WHERE newbie NOT IN (0, 1))
            AS invalid_newbie,

        COUNT(*) FILTER (WHERE visit NOT IN (0, 1))
            AS invalid_visit,

        COUNT(*) FILTER (WHERE conversion NOT IN (0, 1))
            AS invalid_conversion

    FROM hillstrom_raw
""").df()

In [ ]:
con.sql("""
    WITH duplicates AS (
        SELECT
            *,
            COUNT(*) AS occurrences
        FROM hillstrom_raw
        GROUP BY ALL
    )

    SELECT
        SUM(occurrences - 1)
            AS duplicate_rows,

        COUNT(*) FILTER (WHERE occurrences > 1)
            AS duplicate_groups

    FROM duplicates
""").df()

We can't decide to deplete all 6562 rows just because they are duplicate. Since this dataset has no unique id, we have to investigate more by visualization.

In [ ]:
duplicates = con.sql("""
    SELECT
        *,
        COUNT(*) AS occurrences
    FROM hillstrom_raw
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY occurrences DESC
""").df()

display(duplicates.head(15))

In [ ]:
import matplotlib.pyplot as plt

spend_plot = (
    duplicates.groupby("history", as_index=False)["occurrences"]
    .sum()
    .sort_values("history")
)

plt.figure(figsize=(10, 5))
plt.bar(spend_plot["history"], spend_plot["occurrences"])
plt.title("Duplicate Records by Spend")
plt.xlabel("history spending")
plt.ylabel("Total Occurrences")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

duplicates["extra_rows"] = duplicates["occurrences"] - 1

duplicates["spend_group"] = np.where(
    duplicates["spend"] == 0,
    "Spend = 0",
    "Spend > 0"
)

plot_data = duplicates.pivot_table(
    index="segment",
    columns="spend_group",
    values="extra_rows",
    aggfunc="sum",
    fill_value=0
)

plot_data.plot(
    kind="bar",
    stacked=True,
    figsize=(9, 5)
)

plt.title("Duplicate Rows by Treatment Group and Spending")
plt.xlabel("Treatment Group")
plt.ylabel("Number of Extra Duplicate Rows")
plt.xticks(rotation=0)
plt.legend(title="Spending")
plt.tight_layout()
plt.show()

## Hypothesis 1: Mens E-Mail vs Control

**Research Question:** Does sending a men's promotional email affect customer revenue?

- H₀: μ_M = μ_C
- H₁: μ_M ≠ μ_C

## Hypothesis 2: Womens E-Mail vs Control

**Research Question:** Does sending a women's promotional email affect customer revenue?

- H₀: μ_W = μ_C
- H₁: μ_W ≠ μ_C

## Hypothesis 3: Mens E-Mail vs Womens E-Mail

**Research Question:** Do men's and women's promotional emails generate different average customer revenue?

- H₀: μ_M = μ_W
- H₁: μ_M ≠ μ_W

## Statistical Settings

- Primary Metric: Revenue per Customer (RPU)
- Significance Level: α = 0.05
- Test: Two-sided Welch's t-test
- Confidence Level: 95%
- Multiple Testing Correction: Holm Method

In [ ]:
group_counts = con.sql("""
    SELECT 
        segment,
        COUNT(*) as n
        FROM hillstrom_raw
    GROUP BY segment
    ORDER BY segment
""").df()
group_counts.head()

By eye balling the count of each marketing segment, there are approximately 1:1:1. However, I decide to continue use chi-square goodness of fit test at alpha = 0.05 to check for a statistically proven even sample ratio.

In [ ]:
from scipy.stats import chisquare

observed = group_counts["n"].to_numpy()

# Expected equal allocation (1:1:1)
expected = [observed.sum() / 3] * 3

chi2, p_value = chisquare(
    f_obs=observed,
    f_exp=expected
)

print("Chi-square statistic:", chi2)
print("p-value:", p_value)

we fail to reject the null hypothesis since p-value is 0.9 which is much higher than the alpha.

In [ ]:
balance_summary = con.sql("""
    SELECT
        segment,
        COUNT(*) AS n,
        AVG(history) AS avg_history,
        AVG(recency) AS avg_recency,
        AVG(mens) AS pct_mens,
        AVG(womens) AS pct_womens,
        AVG(newbie) AS pct_newbie

    FROM hillstrom_raw
    GROUP BY segment
    ORDER BY segment
""").df()

display(balance_summary)

Just by comparing mean for every variable in the three segments is not enough. We would like to to SMD(standardized mean differences) to have some statistical significnat in our conclusion about covariable balances.

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations

df = con.sql("""
    SELECT
        segment,
        history,
        recency,
        mens,
        womens,
        newbie
    FROM hillstrom_raw
""").df()

variables = [
    "history", "recency",
    "mens", "womens", "newbie"
]

groups = [
    "No E-Mail",
    "Mens E-Mail",
    "Womens E-Mail"
]

results = []

for var in variables:
    for a, b in combinations(groups, 2):

        x = df.loc[df["segment"] == a, var]
        y = df.loc[df["segment"] == b, var]

        pooled_sd = np.sqrt(
            (x.var(ddof=1) + y.var(ddof=1)) / 2
        )

        smd = (
            (x.mean() - y.mean()) / pooled_sd
            if pooled_sd > 0 else np.nan
        )

        results.append({
            "variable": var,
            "comparison": f"{a} vs {b}",
            "SMD": smd
        })

smd_results = pd.DataFrame(results)

display(smd_results)

Since all abs(smd) are way below 0.1, I believe that the numerical covariables in three difference segments are balanced.

In [ ]:
from scipy.stats import chi2_contingency

categorical_df = con.sql("""
    SELECT
        segment,
        history_segment,
        zip_code,
        channel
    FROM hillstrom_raw
""").df()

for col in ["history_segment", "zip_code", "channel"]:

    table = pd.crosstab(
        categorical_df["segment"],
        categorical_df[col]
    )

    chi2, p, dof, expected = chi2_contingency(table)

    print(f"{col}: p-value = {p:.4f}")

our null hypothesis is that the categorical variables are all balances at alpha = 0.05. We fail to reject the null hypothesis, so now both numerical and categorical variables are balanced in the three segments.


NOTE: visit, conversion, and spend are all measured during the same two-week period following the email campaign.

In [ ]:
experiment_metrics = con.sql("""
    SELECT
        segment,
        COUNT(*) AS n,
        AVG(spend) AS mean_revenue,
        STDDEV_SAMP(spend) AS std_revenue,
        AVG(conversion) AS conversion_rate,
        AVG(visit) AS visit_rate
    FROM hillstrom_raw
    GROUP BY segment
    ORDER BY segment
""").df()

display(experiment_metrics)

In [ ]:
from scipy.stats import ttest_ind
import pandas as pd

df = con.sql("""
    SELECT segment, spend
    FROM hillstrom_raw
""").df()

control = df.loc[df["segment"] == "No E-Mail", "spend"]
mens = df.loc[df["segment"] == "Mens E-Mail", "spend"]
womens = df.loc[df["segment"] == "Womens E-Mail", "spend"]

In [ ]:
from statsmodels.stats.multitest import multipletests
comparisons = {
    "Mens vs Control": (mens, control),
    "Womens vs Control": (womens, control),
    "Mens vs Womens": (mens, womens)
}

results = []
for name, (a, b) in comparisons.items():
    t_stat, p_value = ttest_ind(a, b, equal_var=False, alternative="two-sided")
    results.append({
        "comparison": name,
        "ATE": a.mean() - b.mean(),
        "t_statistic": t_stat,
        "p_value": p_value,
    })
results_df = pd.DataFrame(results)
results_df["Holm p-value"] = multipletests(
    results_df["p_value"], method="holm"
)[1]
display(results_df)


A/B Testing — Preliminary Results
NOTE: Holm's p-value also gave statistically significancy at alpha = 0.05

The results indicate that both promotional email campaigns significantly increase revenue per customer compared to the control group. Mens E-Mail generates the largest incremental revenue ($0.77 per customer), followed by Womens E-Mail ($0.42 per customer). Additionally, Mens E-Mail significantly outperforms Womens E-Mail ($0.35 per customer).


These findings suggest that Mens E-Mail is the more effective campaign on average. However, further analysis is needed to evaluate the precision of these estimates and whether personalized targeting can improve marketing performance.

In [ ]:
df = con.sql("""
    SELECT
        segment,
        history,
        spend
    FROM hillstrom_raw
""").df()

theta = df["history"].cov(df["spend"]) / df["history"].var()

print("CUPED theta:", theta)

In [ ]:
df["spend_cuped"] = (
    df["spend"]
    - theta * (df["history"] - df["history"].mean())
)

In [ ]:
df.groupby("segment")[
    ["spend", "spend_cuped"]
].mean()

In [ ]:
original_var = df["spend"].var()
cuped_var = df["spend_cuped"].var()

variance_reduction = 1 - (cuped_var / original_var)

print(f"Original Variance: {original_var:.4f}")
print(f"CUPED Variance: {cuped_var:.4f}")
print(f"Variance Reduction: {variance_reduction:.2%}")

In [ ]:
correlation = df["history"].corr(df["spend"])

print(f"Correlation: {correlation:.4f}")
print(f"R-squared: {correlation**2:.4%}")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

df = con.sql("""
    SELECT segment, history, spend
    FROM hillstrom_raw
""").df()

# CUPED adjustment
theta = df["history"].cov(df["spend"]) / df["history"].var()

df["spend_cuped"] = (
    df["spend"] - theta * (df["history"] - df["history"].mean())
)

comparisons = {
    "Mens vs Control": ("Mens E-Mail", "No E-Mail"),
    "Womens vs Control": ("Womens E-Mail", "No E-Mail"),
    "Mens vs Womens": ("Mens E-Mail", "Womens E-Mail")
}

cuped_results = []

for metric in ["spend", "spend_cuped"]:
    for name, (group_a, group_b) in comparisons.items():

        a = df.loc[df["segment"] == group_a, metric]
        b = df.loc[df["segment"] == group_b, metric]

        ate = a.mean() - b.mean()

        se = np.sqrt(
            a.var(ddof=1) / len(a)
            + b.var(ddof=1) / len(b)
        )

        test = ttest_ind(a, b, equal_var=False)
        ci = test.confidence_interval(confidence_level=0.95)

        cuped_results.append({
            "Method": "CUPED" if metric == "spend_cuped" else "Original",
            "Comparison": name,
            "ATE": ate,
            "SE": se,
            "CI Lower": ci.low,
            "CI Upper": ci.high,
            "CI Width": ci.high - ci.low
        })

cuped_table = pd.DataFrame(cuped_results)

display(cuped_table.round(4))

CUPED adjustment using historical spending reduced overall outcome variance by only 0.05%, with negligible improvements in treatment-effect standard errors and confidence interval widths. The estimated treatment effects remained largely unchanged, suggesting that historical spending alone provides limited value for variance reduction in this experiment. I decide to use multivariable ancova to see if multiple predictors could explain spending.

In [ ]:
import statsmodels.formula.api as smf

df = con.sql("SELECT * FROM hillstrom_raw").df()

ancova_model = smf.ols(
    """
    spend ~ C(segment, Treatment(reference='No E-Mail'))
          + history
          + recency
          + mens
          + womens
          + newbie
          + C(channel)
          + C(zip_code)
    """,
    data=df
).fit(cov_type="HC3")

In [ ]:
import statsmodels.formula.api as smf

# Baseline model: treatment only
baseline = smf.ols(
    "spend ~ C(segment)",
    data=df
).fit()

# Full ANCOVA model already stored in 'model'
variance_reduction = (
    1 - ancova_model.ssr / baseline.ssr
)


print(f"ANCOVA Variance Reduction: {variance_reduction:.4%}")

In [ ]:
# Define explicit group comparisons and corresponding ANCOVA coefficients.
m = "C(segment, Treatment(reference='No E-Mail'))[T.Mens E-Mail]"
w = "C(segment, Treatment(reference='No E-Mail'))[T.Womens E-Mail]"
comparison_terms = {
    "Mens vs Control": ("Mens E-Mail", "No E-Mail", {m: 1}),
    "Womens vs Control": ("Womens E-Mail", "No E-Mail", {w: 1}),
    "Mens vs Womens": ("Mens E-Mail", "Womens E-Mail", {m: 1, w: -1}),
}
results = []
for name, (group_a, group_b, terms) in comparison_terms.items():
    a = df.loc[df["segment"] == group_a, "spend"]
    b = df.loc[df["segment"] == group_b, "spend"]
    original_se = np.sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    contrast = np.zeros(len(ancova_model.params))
    for term, coefficient in terms.items():
        contrast[ancova_model.params.index.get_loc(term)] = coefficient
    ancova_se = float(np.asarray(ancova_model.t_test(contrast).sd).item())
    results.append({
        "Comparison": name,
        "Original SE": original_se,
        "ANCOVA SE": ancova_se,
        "SE Reduction (%)": (1 - ancova_se / original_se) * 100
    })
display(pd.DataFrame(results).round(4))


I realized that even multivariable ancova can't yield a good correlation corresponding to spending. Therefore, I decide to move to A/A testing and see the performance of these optimizations.

In [ ]:
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

control = con.sql("""
    SELECT history, spend
    FROM hillstrom_raw
    WHERE segment = 'No E-Mail'
""").df()

# CUPED adjustment
theta = control["history"].cov(control["spend"]) / control["history"].var()

control["spend_cuped"] = (
    control["spend"]
    - theta * (control["history"] - control["history"].mean())
)

rng = np.random.default_rng(42)
n_simulations = 1000

false_positives = {"Original": 0, "CUPED": 0}

for _ in range(n_simulations):

    indices = rng.permutation(len(control))
    groups = np.array_split(indices, 3)

    for method, outcome in [
        ("Original", "spend"),
        ("CUPED", "spend_cuped")
    ]:
        values = control[outcome].to_numpy()

        p_values = [
            ttest_ind(
                values[groups[a]],
                values[groups[b]],
                equal_var=False
            ).pvalue
            for a, b in [(0, 1), (0, 2), (1, 2)]
        ]

        reject, _, _, _ = multipletests(
            p_values, alpha=0.05, method="holm"
        )

        if any(reject):
            false_positives[method] += 1

for method, count in false_positives.items():
    print(f"{method} FWER: {count / n_simulations:.2%}")

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

control = con.sql("""
    SELECT *
    FROM hillstrom_raw
    WHERE segment = 'No E-Mail'
""").df()

rng = np.random.default_rng(42)
n_simulations = 1000
ancova_false_positives = 0

for _ in range(n_simulations):

    indices = rng.permutation(len(control))
    groups = np.array_split(indices, 3)

    # Create three virtual treatment groups
    labels = np.empty(len(control), dtype="<U1")

    for label, idx in zip(["A", "B", "C"], groups):
        labels[idx] = label

    control["aa_group"] = labels

    # Multivariable ANCOVA
    model = smf.ols(
        """
        spend ~ C(aa_group)
              + history
              + recency
              + mens
              + womens
              + newbie
              + C(channel)
              + C(zip_code)
        """,
        data=control
    ).fit(cov_type="HC3")

    # Three pairwise comparisons
    p_values = [
        model.pvalues["C(aa_group)[T.B]"],
        model.pvalues["C(aa_group)[T.C]"],
        float(model.t_test(
            "C(aa_group)[T.B] = C(aa_group)[T.C]"
        ).pvalue)
    ]

    reject, _, _, _ = multipletests(
        p_values,
        alpha=0.05,
        method="holm"
    )

    if any(reject):
        ancova_false_positives += 1

ancova_fwer = ancova_false_positives / n_simulations
print("Original FWER: 3.10%")  # Previous result
print("CUPED FWER: 3.10%")     # Previous result
print(f"ANCOVA FWER: {ancova_fwer:.2%}")

Both CUPED and multivariable ANCOVA were evaluated as variance-reduction techniques. However, neither method meaningfully improved treatment-effect precision. The original unadjusted A/B test was therefore retained as the primary analysis, with adjusted estimates reported as robustness checks.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import norm

power_df = con.sql("SELECT * FROM hillstrom_raw").df()

# CUPED
theta = (
    power_df["history"].cov(power_df["spend"])
    / power_df["history"].var()
)

power_df["spend_cuped"] = (
    power_df["spend"]
    - theta * (power_df["history"] - power_df["history"].mean())
)

# Multivariable ANCOVA
power_ancova = smf.ols(
    """
    spend ~ C(segment, Treatment(reference='No E-Mail'))
          + history + recency + mens + womens + newbie
          + C(channel) + C(zip_code)
    """,
    data=power_df
).fit(cov_type="HC3")

In [ ]:
groups = {
    "Mens vs Control": ("Mens E-Mail", "No E-Mail"),
    "Womens vs Control": ("Womens E-Mail", "No E-Mail"),
    "Mens vs Womens": ("Mens E-Mail", "Womens E-Mail")
}

def get_se(a, b):
    return np.sqrt(
        a.var(ddof=1) / len(a)
        + b.var(ddof=1) / len(b)
    )

m = "C(segment, Treatment(reference='No E-Mail'))[T.Mens E-Mail]"
w = "C(segment, Treatment(reference='No E-Mail'))[T.Womens E-Mail]"

contrasts = {
    "Mens vs Control": {m: 1},
    "Womens vs Control": {w: 1},
    "Mens vs Womens": {m: 1, w: -1}
}

se_results = []

for name, (group_a, group_b) in groups.items():

    a = power_df[power_df["segment"] == group_a]
    b = power_df[power_df["segment"] == group_b]

    # Original and CUPED
    original_se = get_se(a["spend"], b["spend"])
    cuped_se = get_se(a["spend_cuped"], b["spend_cuped"])

    # ANCOVA robust SE
    contrast = np.zeros(len(power_ancova.params))

    for term, coefficient in contrasts[name].items():
        contrast[power_ancova.params.index.get_loc(term)] = coefficient

    ancova_se = float(np.asarray(
        power_ancova.t_test(contrast).sd
    ).item())

    for method, se in [
        ("Original", original_se),
        ("CUPED", cuped_se),
        ("ANCOVA", ancova_se)
    ]:
        se_results.append({
            "Comparison": name,
            "Method": method,
            "SE": se
        })

se_df = pd.DataFrame(se_results)

In [ ]:
alpha = 0.05 / 3

z_critical = norm.ppf(1 - alpha / 2)

def calculate_power(effect, se):
    return (
        norm.sf(z_critical - effect / se)
        + norm.cdf(-z_critical - effect / se)
    )

power_results = se_df.copy()

power_results["MDE 80%"] = (
    (z_critical + norm.ppf(0.80)) * power_results["SE"]
)

power_results["MDE 90%"] = (
    (z_critical + norm.ppf(0.90)) * power_results["SE"]
)

power_results["Power at $0.50 (%)"] = power_results["SE"].apply(
    lambda se: calculate_power(0.50, se) * 100
)

display(power_results.round(4))

Multivariable ANCOVA achieved marginally higher statistical power and lower MDE than CUPED and the unadjusted estimator. However, the improvements were negligible, supporting the use of the simpler unadjusted A/B test as the primary analysis.

Bonferonni makes alpha = alpha / n, where n equals the number of experiement tested in the ab test. In our case it's three, men vs control, women vs control, men vs women.
I previously used holm's regulation which did alpha = alpha/3 and alpha/2 and alpha/1, which alpha /3 for the smallest p-value and so on. 

For MDE and power analysis, a conservative Bonferroni-adjusted significance level (α = 0.05/3) was used because it provides a consistent and comparable detection threshold across all three estimation methods.

After we compute the result of our ab test and realized that both two new treatments are effective for the population, we decide to do a HTE(heterogenous treatment effect) analysis to see which sample is most attracted by the two treatments.

In [ ]:
import pandas as pd
import numpy as np

features = ["history_segment", "zip_code", "channel"]

hte_tables = {}

for feature in features:

    # SQL: conversion rates by subgroup and treatment
    summary = con.sql(f"""
        SELECT
            {feature} AS customer_group,
            segment,
            COUNT(*) AS n,
            AVG(conversion) AS conversion_rate

        FROM hillstrom_raw
        GROUP BY {feature}, segment
    """).df()

    rates = summary.pivot(
        index="customer_group",
        columns="segment",
        values="conversion_rate"
    )

    counts = summary.pivot(
        index="customer_group",
        columns="segment",
        values="n"
    )

    result = pd.DataFrame({
        "Control N": counts["No E-Mail"],
        "Mens N": counts["Mens E-Mail"],
        "Womens N": counts["Womens E-Mail"],

        "Control CVR (%)": rates["No E-Mail"] * 100,
        "Mens CVR (%)": rates["Mens E-Mail"] * 100,
        "Womens CVR (%)": rates["Womens E-Mail"] * 100,

        "Mens Uplift (pp)": (
            rates["Mens E-Mail"] - rates["No E-Mail"]
        ) * 100,

        "Womens Uplift (pp)": (
            rates["Womens E-Mail"] - rates["No E-Mail"]
        ) * 100
    })

    # Best estimated action, allowing No E-Mail
    options = np.column_stack([
        np.zeros(len(result)),
        result["Mens Uplift (pp)"],
        result["Womens Uplift (pp)"]
    ])

    campaigns = np.array([
        "No E-Mail", "Mens E-Mail", "Womens E-Mail"
    ])

    result["Best Estimated Campaign"] = campaigns[
        np.argmax(options, axis=1)
    ]

    result["Max Uplift (pp)"] = options.max(axis=1)

    result = result.sort_values(
        "Max Uplift (pp)", ascending=False
    )

    hte_tables[feature] = result

    print(f"\n===== {feature} =====")
    display(result.round(3))

In [ ]:
import matplotlib.pyplot as plt

for feature, table in hte_tables.items():

    plot_data = table[
        ["Mens Uplift (pp)", "Womens Uplift (pp)"]
    ].sort_index()

    ax = plot_data.plot(
        kind="barh",
        figsize=(10, max(4, len(plot_data) * 0.8))
    )

    ax.axvline(0, color="black", linestyle="--")

    plt.title(f"Conversion Uplift by {feature}")
    plt.xlabel("Incremental Conversion Rate (pp)")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

hte_tests = []

for feature in features:

    model = smf.ols(
        f"""
        conversion ~
        C(segment, Treatment(reference='No E-Mail'))
        * C({feature})
        """,
        data=con.sql("SELECT * FROM hillstrom_raw").df()
    ).fit(cov_type="HC3")

    interaction_terms = [
        name for name in model.params.index
        if ":" in name
    ]

    R = np.zeros((len(interaction_terms), len(model.params)))

    for i, term in enumerate(interaction_terms):
        R[i, model.params.index.get_loc(term)] = 1

    test = model.wald_test(R, scalar=True)

    hte_tests.append({
        "Feature": feature,
        "Interaction p-value": float(test.pvalue)
    })

hte_tests = pd.DataFrame(hte_tests)

hte_tests["Holm p-value"] = multipletests(
    hte_tests["Interaction p-value"],
    method="holm"
)[1]

display(hte_tests.round(4))

From the result above, we can tell that we can't make and conclusions about zip_code and channel because they aren't statistically significant. Only history_segment is statisticallys significant and some history spending range conversion rate were dominated by female, and some by male.

In [ ]:
import pandas as pd

features = ["history_segment", "zip_code", "channel"]
revenue_tables = {}

for feature in features:

    summary = con.sql(f"""
        SELECT
            {feature} AS customer_group,
            segment,
            COUNT(*) AS n,
            AVG(spend) AS avg_revenue
        FROM hillstrom_raw
        GROUP BY {feature}, segment
    """).df()

    revenue = summary.pivot(
        index="customer_group",
        columns="segment",
        values="avg_revenue"
    )

    counts = summary.pivot(
        index="customer_group",
        columns="segment",
        values="n"
    )

    result = pd.DataFrame({
        "Control N": counts["No E-Mail"],
        "Mens N": counts["Mens E-Mail"],
        "Womens N": counts["Womens E-Mail"],

        "Control RPU": revenue["No E-Mail"],
        "Mens RPU": revenue["Mens E-Mail"],
        "Womens RPU": revenue["Womens E-Mail"],

        "Mens Revenue Uplift": (
            revenue["Mens E-Mail"] - revenue["No E-Mail"]
        ),

        "Womens Revenue Uplift": (
            revenue["Womens E-Mail"] - revenue["No E-Mail"]
        )
    })

    revenue_tables[feature] = result

    print(f"\n===== {feature} =====")
    display(result.round(4))
    

In [ ]:
import matplotlib.pyplot as plt

for feature, table in revenue_tables.items():

    plot_data = table[
        ["Mens Revenue Uplift", "Womens Revenue Uplift"]
    ].sort_index()

    ax = plot_data.plot(
        kind="barh",
        figsize=(10, 5)
    )

    ax.axvline(0, color="black", linestyle="--")

    plt.title(f"Revenue Uplift by {feature}")
    plt.xlabel("Incremental Revenue per Customer ($)")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

spend_hte_df = con.sql("SELECT * FROM hillstrom_raw").df()

hte_results = []

for feature in features:

    model = smf.ols(
        f"""
        spend ~
        C(segment, Treatment(reference='No E-Mail'))
        * C({feature})
        """,
        data=spend_hte_df
    ).fit(cov_type="HC3")

    interaction_terms = [
        name for name in model.params.index
        if ":" in name
    ]

    R = np.zeros((len(interaction_terms), len(model.params)))

    for i, term in enumerate(interaction_terms):
        R[i, model.params.index.get_loc(term)] = 1

    test = model.wald_test(R, scalar=True)

    hte_results.append({
        "Feature": feature,
        "p_value": float(test.pvalue)
    })

hte_results = pd.DataFrame(hte_results)

hte_results["Holm p-value"] = multipletests(
    hte_results["p_value"],
    method="holm"
)[1]

display(hte_results.round(4))

### Heterogeneous Treatment Effects (HTE) — Conclusion

The HTE analysis found statistically significant differences in conversion uplift across historical spending segments (Holm-adjusted p = 0.0326), while geographic location and purchasing channel showed no significant heterogeneity. However, none of these variables demonstrated statistically significant heterogeneity in incremental revenue after Holm correction (all adjusted p-values > 0.05). These findings suggest that historical spending may influence how customers respond to promotional emails in terms of conversion, but the evidence is insufficient to conclude that these differences translate into higher incremental revenue. Therefore, by looing at the current results, we could conclude that the corresponding website/platform would need to be more attracted the customers. Further uplift modeling, followed by out-of-sample evaluation, is needed to determine whether personalized targeting can improve marketing performance.

I used a T-Learner with Ridge Regression to estimate personalized treatment effects from a three-arm randomized experiment. I trained separate outcome models for each treatment group, then compared their predictions to estimate the incremental revenue from each email campaign. I selected Ridge because it is interpretable and reduces overfitting, allowing me to focus on causal policy evaluation rather than model complexity.

In [ ]:
from sklearn.model_selection import train_test_split

df_model = con.sql("SELECT * FROM hillstrom_raw").df()

# Fixed random seed and stratification maintain the treatment-arm mix.
train_df, temp_df = train_test_split(
    df_model, test_size=0.40, random_state=42, stratify=df_model["segment"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df["segment"]
)
print(f"Train: {len(train_df):,} | Validation: {len(val_df):,} | Test: {len(test_df):,}")


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Pre-treatment features only
numerical_features = [
    "history", "recency", "mens", "womens", "newbie"
]

categorical_features = [
    "history_segment", "channel", "zip_code"
]

features = numerical_features + categorical_features

# 5-fold CV
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Candidate regularization strengths
alphas = [0.1, 1, 10, 100, 1000]

In [ ]:
t_models = {}
cv_results = []

groups = ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]

for group in groups:

    # Select training observations for this treatment
    group_df = train_df[train_df["segment"] == group]

    X = group_df[features]
    y = group_df["spend"]

    # Preprocessing
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"),
         categorical_features)
    ])

    # Ridge pipeline
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("ridge", Ridge())
    ])

    # Grid search with 5-fold CV
    search = GridSearchCV(
        pipeline,
        param_grid={"ridge__alpha": alphas},
        cv=cv,
        scoring="neg_mean_squared_error",
        n_jobs=-1
    )

    search.fit(X, y)

    # Save best fitted model
    t_models[group] = search.best_estimator_

    cv_results.append({
        "Treatment": group,
        "Best Alpha": search.best_params_["ridge__alpha"],
        "CV MSE": -search.best_score_
    })

cv_table = pd.DataFrame(cv_results)

display(cv_table)

Using spending as y tends to be useless, even sending mens email to all people outperforms the model. This could be forecasted due to the previous hte analysis which spending as y was statistiaclly insignificant

In [ ]:
# Supplementary: conversion-optimized Logistic T-Learner on the validation set.
# This section was incomplete in the original notebook, so training is shown explicitly.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

cv_conversion = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
conversion_models = {}
for group in ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]:
    subset = train_df.loc[train_df["segment"] == group]
    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ])
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("logistic", LogisticRegression(max_iter=2000))
    ])
    grid = GridSearchCV(
        pipe, {"logistic__C": [0.01, 0.1, 1, 10]},
        cv=cv_conversion, scoring="neg_log_loss", n_jobs=-1
    )
    grid.fit(subset[features], subset["conversion"])
    conversion_models[group] = grid.best_estimator_

conversion_preds = pd.DataFrame(index=val_df.index)
for group in ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]:
    conversion_preds[group] = conversion_models[group].predict_proba(
        val_df[features]
    )[:, 1]
conversion_preds["Recommended Treatment"] = conversion_preds[
    ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]
].idxmax(axis=1)

actual_treatment = val_df["segment"].to_numpy()
actual_conversion = val_df["conversion"].to_numpy()
recommended = conversion_preds["Recommended Treatment"].to_numpy()

def ipw_conversion(policy):
    return np.mean(3 * (actual_treatment == policy) * actual_conversion)

strategies = {
    "No E-Mail to Everyone": np.full(len(val_df), "No E-Mail"),
    "Mens E-Mail to Everyone": np.full(len(val_df), "Mens E-Mail"),
    "Womens E-Mail to Everyone": np.full(len(val_df), "Womens E-Mail"),
    "Personalized T-Learner": recommended
}
results = pd.DataFrame({
    "Strategy": list(strategies),
    "Estimated Conversion Rate (%)": [
        100 * ipw_conversion(p) for p in strategies.values()
    ]
})
display(results.round(4))


In [ ]:
import numpy as np
import pandas as pd

from econml.dml import CausalForestDML
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyClassifier

# Prepare features
X_cf_train = pd.get_dummies(
    train_df[features],
    columns=categorical_features,
    dtype=float
)

X_cf_test = pd.get_dummies(
    test_df[features],
    columns=categorical_features,
    dtype=float
).reindex(columns=X_cf_train.columns, fill_value=0)

Y_train = train_df["spend"].to_numpy()
T_train = train_df["segment"].to_numpy()

# Causal Forest for revenue HTE
spend_forest = CausalForestDML(
    model_y=Ridge(alpha=1000),
    model_t=DummyClassifier(strategy="prior"),

    discrete_outcome=False,
    discrete_treatment=True,

    categories=[
        "No E-Mail",
        "Mens E-Mail",
        "Womens E-Mail"
    ],

    n_estimators=300,
    min_samples_leaf=200,
    max_depth=8,

    cv=3,
    honest=True,
    random_state=42,
    n_jobs=-1
)

spend_forest.fit(
    Y_train,
    T_train,
    X=X_cf_train.to_numpy()
)

In [ ]:
# Estimate revenue CATE
cf_spend_preds = pd.DataFrame(index=test_df.index)

cf_spend_preds["No E-Mail"] = 0.0

cf_spend_preds["Mens E-Mail"] = spend_forest.effect(
    X_cf_test.to_numpy(),
    T0="No E-Mail",
    T1="Mens E-Mail"
)

cf_spend_preds["Womens E-Mail"] = spend_forest.effect(
    X_cf_test.to_numpy(),
    T0="No E-Mail",
    T1="Womens E-Mail"
)

# Recommend highest incremental revenue
cf_spend_recommended = cf_spend_preds.idxmax(axis=1).to_numpy()

display(cf_spend_preds.head())
display(pd.Series(cf_spend_recommended).value_counts())

In [ ]:
from scipy.stats import norm

def evaluate_spend_policy(recommended, model_name):

    actual_treatment = test_df["segment"].to_numpy()
    actual_spend = test_df["spend"].to_numpy()

    recommended = np.asarray(recommended)

    # Assume 1/3 randomized assignment probability
    cf_values = 3 * (
        actual_treatment == recommended
    ) * actual_spend

    mens_values = 3 * (
        actual_treatment == "Mens E-Mail"
    ) * actual_spend

    # Estimated revenue per customer
    personalized_revenue = cf_values.mean()
    mens_revenue = mens_values.mean()

    # Paired IPW difference
    difference = cf_values - mens_values

    improvement = difference.mean()
    se = difference.std(ddof=1) / np.sqrt(len(difference))

    # 95% CI and two-sided p-value
    ci_lower = improvement - 1.96 * se
    ci_upper = improvement + 1.96 * se

    if se > 0:
        z = improvement / se
        p_value = 2 * norm.sf(abs(z))
    else:
        p_value = 1.0 if improvement == 0 else np.nan

    print(f"\n===== {model_name} TEST RESULTS =====")
    print(f"All Mens Revenue: ${mens_revenue:.4f}")
    print(f"Personalized Revenue: ${personalized_revenue:.4f}")
    print(f"Improvement: ${improvement:+.4f}")
    print(f"95% CI: [${ci_lower:.4f}, ${ci_upper:.4f}]")
    print(f"P-value: {p_value:.4f}")

    return {
        "Model": model_name,
        "All Mens RPU": mens_revenue,
        "Personalized RPU": personalized_revenue,
        "Improvement": improvement,
        "CI Lower": ci_lower,
        "CI Upper": ci_upper,
        "P-value": p_value
    }

In [ ]:
cf_result = evaluate_spend_policy(
    cf_spend_recommended,
    "Causal Forest"
)

In [ ]:
# Prepare test features
X_test = test_df[features]

# Predict spending under each treatment
t_spend_preds = pd.DataFrame(index=test_df.index)

groups = ["No E-Mail", "Mens E-Mail", "Womens E-Mail"]

for group in groups:
    t_spend_preds[group] = t_models[group].predict(X_test)

# Recommend treatment with highest predicted revenue
t_spend_recommended = t_spend_preds.idxmax(axis=1).to_numpy()

display(t_spend_preds.head())
display(pd.Series(t_spend_recommended).value_counts())

In [ ]:
t_result = evaluate_spend_policy(
    t_spend_recommended,
    "Ridge T-Learner"
)